import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.nn import MSELoss
from sklearn.preprocessing import OneHotEncoder

In [4]:
df=pd.read_csv("california_house_price_dataset.zip")

In [5]:
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [6]:
df["ocean_proximity"].value_counts()

ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: count, dtype: int64

In [18]:
df.shape
df.head(1)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,1.069285,0.90298,0.788462,0.02238,0.020016,0.009024,0.020717,0.55501,0.905198,NEAR BAY


In [17]:
df.iloc[:,:-1].astype(float)
for i in range(9):
    df.iloc[:,i]=df.iloc[:,i]/df.iloc[:,i].max()

In [19]:
X=df.drop("median_house_value",axis=1)
y=df["median_house_value"]

In [20]:
X.head(1)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
0,1.069285,0.90298,0.788462,0.02238,0.020016,0.009024,0.020717,0.55501,NEAR BAY


In [21]:
ohe=OneHotEncoder(drop="first")

In [42]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)

In [43]:
X_val,X_test,y_val,y_test=train_test_split(X_test,y_test,test_size=0.5,random_state=42)
X_val.head(1)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
14100,1.02432,0.78093,0.596154,0.031409,0.060047,0.022869,0.065275,0.103446,NEAR OCEAN


In [44]:
X_train_new=ohe.fit_transform(X_train.iloc[:,[-1]]).toarray() # here [-1] represents a dataframe, because just -1 was series

In [45]:
X_val_new=ohe.transform(X_val.iloc[:,[-1]]).toarray()
X_test_new=ohe.transform(X_test.iloc[:,[-1]]).toarray()

In [46]:
X_train.drop("ocean_proximity",axis=1,inplace=True)
X_val.drop("ocean_proximity",axis=1,inplace=True)
X_test.drop("ocean_proximity",axis=1,inplace=True)

In [47]:
X_train=np.hstack((X_train.values,X_train_new))
X_val=np.hstack((X_val.values,X_val_new))
X_test=np.hstack((X_test.values,X_test_new))


In [48]:
X_train[0][:]
type(y_train)

pandas.core.series.Series

In [49]:
y_train=y_train.to_numpy()
y_val=y_val.to_numpy()
y_test=y_test.to_numpy()

In [52]:
class dataset(Dataset):
    def __init__(self,X,y):
        self.X=torch.tensor(X,dtype=torch.float32)
        self.y=torch.tensor(y,dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self,index):
        return self.X[index],self.y[index]

train_data=dataset(X_train,y_train)
val_data=dataset(X_val,y_val)
test_data=dataset(X_test,y_test)

In [53]:
train_loader=DataLoader(train_data,batch_size=32,shuffle=True)
val_loader=DataLoader(val_data,batch_size=32,shuffle=True)
test_data=DataLoader(test_data,batch_size=32,shuffle=True)

In [64]:
class Model_o(nn.Module):
    def __init__(self):
        super(Model_o,self).__init__()
        self.input_layer=nn.Linear(12,20)
        self.hidden_layer=nn.ReLU() # rectified linear unit (ReLU)
        self.output_layer=nn.Linear(20,1) # unlike in binary classification which had an activation function , regression only has a loss calculator 
                                          # and updates the parameters on the basis of that loss function
    def forward(self,x):
        x=self.input_layer(x)
        x=self.hidden_layer(x)
        x=self.output_layer(x)
        return x

In [65]:
model=Model_o()
criterion=MSELoss()
optimizer=Adam(model.parameters(), lr = 0.01)

In [69]:
epochs=10
for epoch in range(epochs):
    train_loss=0 # accuracy is not calculated in regression
    val_loss=0
    for inputs,labels in train_loader:
        y_pred=model(inputs)
        batch_loss=criterion(y_pred,labels)
        batch_loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        train_loss+=batch_loss.item()
    print(f" train loss = {train_loss}")
    with torch.no_grad():
        for inputs,labels in val_loader:
            y_pred=model(inputs)
            batch_loss=criterion(y_pred,labels)
            val_loss+=batch_loss.item()
    print(f" val loss = {val_loss}")

 train loss = 24.30206846818328
 val loss = nan
 train loss = 24.315249163657427
 val loss = nan
 train loss = 24.32518703304231
 val loss = nan
 train loss = 24.289239993318915
 val loss = nan
 train loss = 24.325991502031684
 val loss = nan
 train loss = 24.314366286620498
 val loss = nan
 train loss = 24.302808871492743
 val loss = nan
 train loss = 24.31557821854949
 val loss = nan
 train loss = 24.313352378085256
 val loss = nan
 train loss = 24.330768909305334
 val loss = nan


In [72]:
for inputs,labels in val_loader:
    print(torch.isnan(inputs).any().sum())
    print(torch.isnan(labels).any())
    break

tensor(1)
tensor(False)


## The Reason why this model isnt learning very well is because of the following reasons:
1. Bad Scaling, i devided each column with its maximum value ( should've used standard scalar)
2. High learning rate is making the learning process unstable
3. Multicollinearity between columns, i didnt account for this but its highly likely that this dataset has multicollinearity between columns
4. No clusters formed, longitude and latitude columns should have been devided into clusters

## The Reason for nan validation loss is because of 1 nan value that i didnt drop